In [0]:
'''%run "../00_config/00_config"

In [0]:
'''%run "../../src/utilities/logger"

In [0]:
'''from delta.tables import DeltaTable

dbutils.widgets.text("start_date", "2026-01-01", "Start Ingestion Date")
dbutils.widgets.text("end_date", datetime.date.today().strftime("%Y-%m-%d"), "End Ingestion Date")

start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")
secret_salt = get_secret_salt()

def process_bronze_to_silver():
    # Read Bronze dataset for date range
    bronze_df = (spark.table(TABLE_BRONZE_FULL)
                 .filter((col("ingestion_date") >= start_date) & (col("ingestion_date") <= end_date)))

    # Step 4.2 & 4.5: Flatten, Cast, and Protect PII
    flattened_df = bronze_df.select(
        col("itemId").alias("item_id"),
        # Mask phone numbers and emails in titles
        regexp_replace(col("title"), r"(\d{3}-\d{3}-\d{4}|\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b)", "[REDACTED]").alias("title"),
        col("categories")[0]["categoryId"].alias("category_id"),
        col("categories")[0]["categoryName"].alias("category_name"),
        # Pseudonymize seller username with HMAC SHA-256
        sha2(concat(col("seller.username"), lit(secret_salt)), 256).alias("seller_key"),
        col("seller.feedbackScore").cast("int").alias("feedback_score"),
        col("seller.feedbackPercentage").cast("double").alias("feedback_percentage"),
        col("price.value").cast("decimal(18,2)").alias("price"),
        col("price.currency").alias("currency"),
        col("condition"),
        to_timestamp(col("itemCreationDate")).alias("listing_date"),
        col("itemLocation.country").alias("country"),
        col("itemLocation.stateOrProvince").alias("state"),
        col("itemLocation.city").alias("city"),
        col("batch_id"),
        col("ingestion_timestamp"),
        col("_rescued_data")
    )

    # Step 4.4: Quality Rules & Quarantine Logic
    valid_cond = (
        col("item_id").isNotNull() & 
        col("price").isNotNull() & 
        (col("price") > 0) & 
        col("_rescued_data").isNull()
    )

    quarantined_df = flattened_df.filter(~valid_cond).withColumn(
        "reason_code", 
        expr("""CASE 
            WHEN item_id IS NULL THEN 'MISSING_ITEM_ID'
            WHEN price IS NULL OR price <= 0 THEN 'INVALID_PRICE'
            WHEN _rescued_data IS NOT NULL THEN 'SCHEMA_DRIFT_DETECTED'
            ELSE 'UNKNOWN_VALIDATION_FAILURE' END""")
    ).withColumn("load_timestamp", current_timestamp())

    # Write failures to silver_quarantine
    quarantined_count = quarantined_df.count()
    if quarantined_count > 0:
        quarantined_df.write.format("delta").mode("append").saveAsTable(TABLE_SILVER_QUARANTINE)

    valid_df = flattened_df.filter(valid_cond)

    # Step 4.6: Deduplicate within batch
    window_spec = Window.partitionBy("item_id").orderBy(col("ingestion_timestamp").desc())
    deduped_df = (valid_df
                  .withColumn("rn", row_number().over(window_spec))
                  .filter("rn = 1")
                  .drop("rn")
                  .withColumn("row_hash", sha2(concat(col("price"), col("condition"), col("title")), 256))
                  .withColumn("load_timestamp", current_timestamp()))

    # Step 4.7: MERGE INTO silver_listings
    if not spark.catalog.tableExists(TABLE_SILVER_LISTINGS):
        deduped_df.write.format("delta").saveAsTable(TABLE_SILVER_LISTINGS)
        inserted, updated = deduped_df.count(), 0
    else:
        target_table = DeltaTable.forName(spark, TABLE_SILVER_LISTINGS)
        target_table.alias("t").merge(
            deduped_df.alias("s"),
            "t.item_id = s.item_id"
        ).whenMatchedUpdate(
            condition="t.row_hash <> s.row_hash",
            set={
                "title": "s.title",
                "price": "s.price",
                "condition": "s.condition",
                "row_hash": "s.row_hash",
                "load_timestamp": "s.load_timestamp"
            }
        ).whenNotMatchedInsertAll().execute()

        inserted, updated = get_last_merge_metrics(spark, TABLE_SILVER_LISTINGS)

    return inserted, updated, quarantined_count

run_step(spark, TABLE_LOGS, "Bronze-to-Silver", "INCREMENTAL", TABLE_SILVER_LISTINGS, f"{start_date} to {end_date}", process_bronze_to_silver)

In [0]:
'''%sql
-- Query 1: Verify Duplicate Primary Keys (Must return 0 rows)
SELECT item_id, COUNT(*) 
FROM workspace.ebay_pipeline.silver_listings 
GROUP BY item_id 
HAVING COUNT(*) > 1;

-- Query 2: Verify Audit Log Metrics (Second run must show 0 inserted, 0 updated)
SELECT layer, target_table, status, rows_inserted, rows_updated, start_time 
FROM workspace.ebay_pipeline.pipeline_execution_logs 
ORDER BY start_time DESC LIMIT 2;

In [0]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %run ../00_config/00_config

# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, 
    TimestampType, DoubleType
)
from delta.tables import DeltaTable
from datetime import datetime
import uuid

# Project Salt for SHA-256 PII Hashing (Proposal Section 3.2)
SALT = "EbayMarketplace2026SaltKey!"
start_time = datetime.now()

TABLE_SILVER_LISTINGS = "ebay_pipeline.silver_listings"
TABLE_QUARANTINE = "ebay_pipeline.silver_quarantine_records"
TABLE_LOGS = "ebay_pipeline.pipeline_execution_logs"

# COMMAND ----------
try:
    df_bronze_inc = spark.table("ebay_pipeline.bronze_ebay_listings_incremental")

    # 1. Type Casting & Salted SHA-256 PII Hashing on Seller Handles
    df_transformed = (
        df_bronze_inc
        .select(
            F.col("raw_payload.itemId").alias("item_id"),
            F.col("raw_payload.title").alias("title"),
            F.col("subcategory_partition").alias("subcategory"),
            F.col("raw_payload.price.value").cast(DoubleType()).alias("price"),
            F.col("raw_payload.price.currency").alias("currency"),
            F.col("raw_payload.condition").alias("condition"),
            F.to_timestamp("raw_payload.itemCreationDate").alias("item_creation_timestamp"),
            F.sha2(F.concat(F.col("raw_payload.seller"), F.lit(SALT)), 256).alias("seller_id_hashed"),
            F.current_timestamp().alias("load_timestamp")
        )
    )

    # 2. Quarantine Records Missing Primary Key (Rubric: Quality/Quarantine)
    df_quarantine = df_transformed.filter(F.col("item_id").isNull() | (F.col("item_id") == ""))
    quarantine_count = df_quarantine.count()
    if quarantine_count > 0:
        (
            df_quarantine
            .withColumn("rejection_reason", F.lit("Missing Primary Key item_id"))
            .withColumn("quarantined_at", F.current_timestamp())
            .write.format("delta").mode("append")
            .saveAsTable(TABLE_QUARANTINE)
        )
        print(f"Quarantined {quarantine_count} invalid records into {TABLE_QUARANTINE}")

    # 3. Deduplicate Clean Staging Records on Natural Key
    df_valid = (
        df_transformed
        .filter(F.col("item_id").isNotNull() & (F.col("item_id") != ""))
        .dropDuplicates(["item_id"])
    )

    # 4. Idempotent MERGE INTO (SCD Type 1 Upsert)
    table_exists = spark.catalog.tableExists(TABLE_SILVER_LISTINGS)

    if not table_exists:
        df_valid.write.format("delta").mode("overwrite").saveAsTable(TABLE_SILVER_LISTINGS)
        inserted = df_valid.count()
        updated = 0
    else:
        silver_delta = DeltaTable.forName(spark, TABLE_SILVER_LISTINGS)
        before_count = spark.table(TABLE_SILVER_LISTINGS).count()

        (
            silver_delta.alias("target")
            .merge(
                df_valid.alias("source"),
                "target.item_id = source.item_id"
            )
            .whenMatchedUpdate(set={
                "title": "source.title",
                "price": "source.price",
                "condition": "source.condition",
                "load_timestamp": "source.load_timestamp"
            })
            .whenNotMatchedInsertAll()
            .execute()
        )
        after_count = spark.table(TABLE_SILVER_LISTINGS).count()
        inserted = after_count - before_count
        updated = df_valid.count() - inserted

    end_time = datetime.now()

    # 5. Log Run to Audit Table
    log_schema = StructType([
        StructField("log_id", StringType(), False),
        StructField("layer", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("parameter_processed", StringType(), True),
        StructField("execution_start_time", TimestampType(), False),
        StructField("execution_end_time", TimestampType(), True),
        StructField("status", StringType(), False),
        StructField("rows_inserted", LongType(), True),
        StructField("rows_updated", LongType(), True),
        StructField("error_message", StringType(), True)
    ])

    log_entry = [(
        str(uuid.uuid4()),
        "Bronze-to-Silver",
        TABLE_SILVER_LISTINGS,
        "bronze_incremental_source",
        start_time,
        end_time,
        "SUCCESS",
        inserted,
        updated,
        None
    )]

    spark.createDataFrame(log_entry, schema=log_schema).write.format("delta").mode("append").saveAsTable(TABLE_LOGS)
    print(f"Silver Merge Succeeded: {inserted} inserted, {updated} updated into {TABLE_SILVER_LISTINGS}")

except Exception as e:
    end_time = datetime.now()
    log_entry = [(
        str(uuid.uuid4()),
        "Bronze-to-Silver",
        TABLE_SILVER_LISTINGS,
        "bronze_incremental_source",
        start_time,
        end_time,
        "FAILURE",
        0,
        0,
        str(e)
    )]
    spark.createDataFrame(log_entry, schema=log_schema).write.format("delta").mode("append").saveAsTable(TABLE_LOGS)
    raise e

In [0]:
# COMMAND ----------
# Rubric Verification: Re-run identical batch and verify 0 duplicates
count_before = spark.table("ebay_pipeline.silver_listings").count()
print(f"Silver count before re-run: {count_before}")

# Re-execute Silver logic
(
    silver_delta.alias("target")
    .merge(
        df_valid.alias("source"),
        "target.item_id = source.item_id"
    )
    .whenMatchedUpdate(set={"load_timestamp": "source.load_timestamp"})
    .whenNotMatchedInsertAll()
    .execute()
)

count_after = spark.table("ebay_pipeline.silver_listings").count()
print(f"Silver count after re-run:  {count_after}")

assert count_before == count_after, "Idempotency failed: duplicate rows created!"
print("IDEMPOTENCY PROVEN: Re-running identical load produced 0 duplicate records.")

# Inspect Execution Audit Trail
display(spark.table("ebay_pipeline.pipeline_execution_logs").orderBy("execution_start_time", ascending=False))